In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sha2, year, to_date, when


spark = SparkSession.builder.enableHiveSupport().getOrCreate()
customers_stg_df = spark.read.parquet("/user/sclsepb4532/sclsepb4532_staging.db/customers_stg")

26/11/08 14:51:35 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/08 14:51:37 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/11/08 14:51:37 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.
26/11/08 14:51:37 WARN Utils: Service 'SparkUI' could not bind on port 4042. Attempting port 4043.
26/11/08 14:51:37 WARN Utils: Service 'SparkUI' could not bind on port 4043. Attempting port 4044.
26/11/08 14:51:40 WARN Client: Neither spark.yarn.jars nor spark.yarn.archive is set, falling back to uploading libraries under SPARK_HOME.
26/11/08 14:52:55 WARN DataStreamer: Exception for BP-1364978691-172.31.64.10-1677754292516:blk_1101628372_27896269
java.io.IOException: Bad response ERROR for BP-1364978691-172.31.64.10-1677

In [2]:
dim_customer_df = customers_stg_df.select(
    sha2(col("customer_id"), 256).alias("customer_key"),
    col("city"),
    col("state"),
    when((col("customer_type") == "Online") & (col("is_active") == "Y"), "High Spend Potential")
    .when(col("is_active") == "Y", "Medium Spend Potential")
    .otherwise("Low/Inactive").alias("customer_segment"),
    year(to_date(col("signup_date"), "yyyy-MM-dd")).alias("registration_year")
)

In [3]:
dim_customer_df.write\
.mode("overwrite")\
.option("compression", "snappy")\
.saveAsTable("sclsepb4532_analytics.dim_customer_analytics")

26/11/08 14:55:47 WARN SessionState: METASTORE_FILTER_HOOK will be ignored, since hive.security.authorization.manager is set to instance of HiveAuthorizerFactory.
